In [30]:
from langgraph.graph import  START, END,StateGraph
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [31]:
load_dotenv()

True

In [32]:
model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", api_key=os.getenv('API_KEY'))

In [33]:
class LLMState(TypedDict):
    question: str
    answer: str

In [34]:
def LLM_QA(state: LLMState) -> LLMState:
    # extract the question from the state
    question = state['question']

    # form a prompt for the LLM
    prompt = f"Answer the following question: {question}"


    # ask the question to the LLM
    answer = model.invoke(prompt).content


    # update the state with the answer from the LLM
    state['answer'] = answer

    return state

In [35]:
graph = StateGraph(LLMState)

# add nodes to the graph
graph.add_node("LLM_QA", LLM_QA)

# add edges to the graph
graph.add_edge(START, "LLM_QA")
graph.add_edge("LLM_QA", END)

# compile the graph
workflow = graph.compile()

In [37]:
initial_state = LLMState({'question': "What is the capital of France?", 'answer': ""})
final_state = workflow.invoke(initial_state)
print(final_state['answer'])

[{'type': 'text', 'text': 'The capital of France is **Paris**.', 'extras': {'signature': 'EvsCCvgCAQw51sc9JUMAfpj5vkZNPxogo5fNt8QrDNdEUGuO3CLlT4WOnJ4+FrrhPMEvwYZFv5BfSGIysDOWEYwwlj9cvoxqWQSJTM+NPNFEhu/IN/QWTBXU0y8x3lz5u4nVvCbpccgjf91G2U3rFmwiahqAAJj7SGSFEbB85vqcY55DDo9EVtOUM6wmXaQQlr7HQgulflTxjPPbpqoCSi+WJgkVEyLQfl7TbjMy/b6GscDvjZqZCx7vgl4BKqRb5+ZRkeqP/BKTtLmQUqVuNH+9vhI29RHwTTlpeFyTl6JOc7iJ2HHPQeK+Fncx/U2kJwhB29rD0y7NymccIrIvH+sLE05hjpHD78emRJkFiau/xGkQboBSAiQNukupTAlGardM+uSAT7M4i/VFRB05ZZ1tKeU0G667O7m7DhqLtxqU4C/t+sTzh8Q06GnO0Qllbd/hyf+vwQXnaRC4VOKBhSVTC87s6m+PBel4GMzcQEMxxqgAosSx30eIL2cQ0w=='}}]
